In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'rank_ratio': [0.0, 0.5, 1.0],
    'tol': [1e-3, 1e-4],
    'max_iter': [5000],
    'n_controls_per_case': [1, 2, 4, 10, 15]

}

metrics = ['c-index', 'c-index-ipcw']


In [ ]:
results_df, summary_df, best_params = ensemble_repeated_grid_search(FastSurvivalSVM, x_train, y_train, param_grid, metrics,
                                  model_name='SSVM Model',
                                  scale_data=True, scaler_type='standard',
                                  random_state=None, n_jobs=None,
                                  n_controls_per_case=None, dataset=subsampling,
                                  feature_columns=feature_columns,
                                  event_col='Regional LN recurrence',
                                  time_col='Survival days',
                                  n_splits=4, n_repeats=5, base_random_state=42,
                                  verbose=True)


RSF MODEL REPEATED GRID SEARCH
Model: FastSurvivalSVM
Scaling: Yes (standard)
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw']
Parameter Grid:
  - alpha: [0.01, 0.1, 1.0, 10.0, 100.0]
  - rank_ratio: [0.0, 0.5, 1.0]
  - tol: [0.001, 0.0001]
  - max_iter: [5000]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total hyperparameter combinations: 150
Folds per repetition: 4
Repetitions: 5
Total evaluations: 3000

[1/150] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=1
  c-index: 0.2377 ± 0.1064
  c-index-ipcw: 0.2924 ± 0.1172
[2/150] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=2
  Elapsed: 0.0min, Est. remaining: 6.2min
  c-index: 0.2732 ± 0.1022
  c-index-ipcw: 0.3373 ± 0.1426
[3/150] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}, n_controls_per_case=4
  Elapsed: 0.1min, Est. remaining: 5.7min
  c-index: 0.3153 ± 0.1245
  c-index-ipcw: 

c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\repeated_cv.py:862: ConvergenceWarning: Optimization did not converge: Warning: Desired error not necessarily achieved due to precision loss.
  model.fit(X_train, y_train)


  c-index: 0.6479 ± 0.1398
  c-index-ipcw: 0.6050 ± 0.1399
[147/150] Testing params: {'alpha': 100.0, 'rank_ratio': 1.0, 'tol': 0.0001, 'max_iter': 5000}, n_controls_per_case=2
  Elapsed: 17.2min, Est. remaining: 0.5min


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\repeated_cv.py:862: ConvergenceWarning: Optimization did not converge: Warning: Desired error not necessarily achieved due to precision loss.
  model.fit(X_train, y_train)


  c-index: 0.6605 ± 0.1332
  c-index-ipcw: 0.6228 ± 0.1698
[148/150] Testing params: {'alpha': 100.0, 'rank_ratio': 1.0, 'tol': 0.0001, 'max_iter': 5000}, n_controls_per_case=4
  Elapsed: 17.4min, Est. remaining: 0.4min
  c-index: 0.6582 ± 0.1306
  c-index-ipcw: 0.6312 ± 0.1319
[149/150] Testing params: {'alpha': 100.0, 'rank_ratio': 1.0, 'tol': 0.0001, 'max_iter': 5000}, n_controls_per_case=10
  Elapsed: 17.7min, Est. remaining: 0.2min
  c-index: 0.7321 ± 0.1095
  c-index-ipcw: 0.6533 ± 0.2112
[150/150] Testing params: {'alpha': 100.0, 'rank_ratio': 1.0, 'tol': 0.0001, 'max_iter': 5000}, n_controls_per_case=15
  Elapsed: 18.0min, Est. remaining: 0.1min
  c-index: 0.7196 ± 0.0936
  c-index-ipcw: 0.6751 ± 0.1270

REPEATED GRID SEARCH COMPLETE
Total time: 18.6 minutes

Best parameters for each metric:
--------------------------------------------------------------------------------

c-index:
  Score: 0.7821 ± 0.0873
  Parameters:
    alpha: 0.01
    rank_ratio: 1.0
    tol: 0.001
    max_

In [8]:
summary_df.to_csv('../../results/repeated_sub/ssvm_sub.csv', index=False)

# Train and Test Final Model with Best Hyperparameters

In [7]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/ssvm_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  alpha: {best_params['alpha']}")
print(f"  rank_ratio: {best_params['rank_ratio']}")
print(f"  tol: {best_params['tol']}")
print(f"  max_iter: {int(best_params['max_iter'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  alpha: 0.01
  rank_ratio: 1.0
  tol: 0.001
  max_iter: 5000
  n_controls_per_case: 2
  c-index-ipcw_mean (CV): 0.7327



In [8]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['n_controls_per_case']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 81
Test samples: 594


In [9]:
# Train final model with best hyperparameters (SVM needs scaling)
scaler = StandardScaler()
x_train_sub_scaled = scaler.fit_transform(x_train_sub)

final_model = FastSurvivalSVM(
    alpha=best_params['alpha'],
    rank_ratio=best_params['rank_ratio'],
    tol=best_params['tol'],
    max_iter=int(best_params['max_iter']),
    random_state=12
)

final_model.fit(x_train_sub_scaled, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [10]:
# Test the model on test set
x_test_scaled = scaler.transform(x_test)

# Calculate c-index
c_index = final_model.score(x_test_scaled, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test_scaled), tau=tau)[0]

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print("  Note: FastSurvivalSVM does not produce survival curves, so Brier score cannot be calculated.")

\nTest Set Performance:
  C-index: 0.7661
  C-index IPCW: 0.7503
  Note: FastSurvivalSVM does not produce survival curves, so Brier score cannot be calculated.


# Feature Importance for SHAP Analysis

In [13]:
# Extract feature importance from SSVM model
# For SVM models, feature importance is based on the absolute value of coefficients
import pandas as pd
import numpy as np

# IMPORTANT: Use the actual feature names from the training data
# The model was trained on x_train_sub_scaled, which came from x_train_sub
actual_features = x_train_sub.columns.tolist()

# Get coefficients from the final SSVM model
# FastSurvivalSVM stores coefficients in coef_ attribute
coefficients = final_model.coef_

# Handle different shapes
if coefficients.ndim > 1:
    coefficients = coefficients.flatten()

# Ensure coefficient length matches the actual features used
if len(coefficients) != len(actual_features):
    print(f"Warning: Coefficient length ({len(coefficients)}) doesn't match feature length ({len(actual_features)})")
    print(f"Using minimum length for mapping")
    min_len = min(len(coefficients), len(actual_features))
    coefficients = coefficients[:min_len]
    actual_features = actual_features[:min_len]

coef_df = pd.DataFrame({
    'Feature': actual_features,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

# Sort by absolute value of coefficient (most important first)
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

# Display top 20 most important features
print("Top 20 Most Important Features for SSVM Model:")
print("="*60)
print(coef_df.head(20).to_string(index=False))
print(f"\nTotal features: {len(coef_df)}")
print(f"Total non-zero coefficients: {(coef_df['Coefficient'] != 0).sum()}")

# Store for SHAP analysis
top_features_ssvm = coef_df.head(20)['Feature'].tolist()
print(f"\n\nTop features stored in 'top_features_ssvm' variable")
print(f"All coefficients stored in 'coef_df' variable")

Top 20 Most Important Features for SSVM Model:
                                                  Feature  Coefficient  Abs_Coefficient
                                       Istotype at CB_1.0     0.285955         0.285955
                      Oestrogen receptor status at CB_1.0     0.273786         0.273786
               Treatment in association with chemotherapy    -0.225116         0.225116
                                       Istotype at CB_4.0    -0.220999         0.220999
                          Adjuvant therapy with bio drugs     0.214042         0.214042
Her2 overexpression (with immunohystochemistry) at CB_1.0    -0.212193         0.212193
                               Radiotherapy on chest wall    -0.175549         0.175549
                      Oestrogen receptor status at CB_0.0    -0.160277         0.160277
                              Endocrine therapy performed    -0.159271         0.159271
                                  Boost dose administered     0.152807   

In [15]:
# Export feature importance to CSV
coef_df.to_csv("../results/features/ssvm_feature_importance.csv", index=False)
print("Feature importance exported to: ../results/features/ssvm_feature_importance.csv")

Feature importance exported to: ../results/features/ssvm_feature_importance.csv
